# 📓 Notebook 21 — LangSmith + LLM Observability---**Phase 7 · Industry Frameworks**> You can't debug what you can't see. LangSmith gives you full visibility into every LLM call, chain step, and agent decision in production.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| Tracing | Record every LLM call with inputs/outputs || Debugging | Trace multi-step agent failures || Evaluation | Systematic quality testing at scale || Datasets | Curated test sets for regression testing || Monitoring | Production dashboards and alerts || Alternatives | OpenTelemetry, Langfuse, Phoenix, Weights & Biases |### 🏗️ Build: Full Observability Pipeline> **Prerequisite:** `pip install langsmith langchain`  > Get your API key at [smith.langchain.com](https://smith.langchain.com)

## 1. Setup

In [ ]:
import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, is_proxy_mode, get_langchain_llm

# LangChain LLM — auto-routes through proxy when available
llm = get_langchain_llm()

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"🤖 Model: {DEFAULT_MODEL} ({mode})")
print(f"   LangChain LLM: {llm.model_name}")

---## 2. Tracing — See Every Step### What Gets Traced AutomaticallyWhen `LANGCHAIN_TRACING_V2=true`, every LangChain operation is traced:```🔍 Trace: "user asked about Python"  ├── PromptTemplate.invoke (2ms)  │     Input: {topic: "Python"}  │     Output: "You are an expert in Python..."  ├── ChatOpenAI.invoke (850ms)  │     Tokens: 45 → 120  │     Cost: $0.0002  │     Output: "Python is a..."  └── StrOutputParser.invoke (0ms)        Output: "Python is a..."```> **Interview Tip:** Tracing is not optional in production. Without it, debugging a multi-step agent failure is like debugging code without logs.

In [ ]:
from langchain_core.prompts import ChatPromptTemplatefrom langchain_core.output_parsers import StrOutputParser# This chain will be automatically traced if LangSmith is configuredprompt = ChatPromptTemplate.from_messages([    ("system", "You are a {role}. Be concise."),    ("human", "{question}"),])chain = prompt | llm | StrOutputParser()# Each invocation creates a trace in LangSmithresult = chain.invoke({"role": "Python expert", "question": "What are list comprehensions?"})print(f"📝 {result[:200]}")print("\n📊 Check LangSmith dashboard for the full trace!")

In [ ]:
# Manual tracing with run names for better organizationfrom langchain_core.runnables import RunnableConfig# Named runs appear clearly in LangSmithresult = chain.invoke(    {"role": "ML engineer", "question": "Explain gradient descent"},    config=RunnableConfig(run_name="gradient_descent_explanation"))print(f"📝 {result[:200]}")# Metadata for filtering tracesresult = chain.invoke(    {"role": "developer", "question": "What is Docker?"},    config=RunnableConfig(        run_name="docker_explanation",        metadata={"user_id": "abhishek", "session": "demo", "difficulty": "beginner"}    ))print(f"📝 {result[:200]}")

---## 3. Evaluation — Systematic Quality Testing

In [ ]:
from langsmith import Clientfrom langsmith.evaluation import evaluate# You need LANGCHAIN_API_KEY set for this sectiontry:    client = Client()    print("✅ LangSmith client connected")except Exception as e:    print(f"⚠️ LangSmith not configured: {e}")    print("   The code below shows the patterns — configure LangSmith to run them")    client = None

In [ ]:
# Create a test datasetif client:    # Create dataset    dataset_name = "agents_course_eval_demo"        try:        dataset = client.create_dataset(dataset_name, description="Evaluation demo for agents course")    except:        # Dataset might already exist        dataset = client.read_dataset(dataset_name=dataset_name)        # Add examples    examples = [        {"input": {"question": "What is Python?"}, "output": {"answer": "Python is a high-level, interpreted programming language created by Guido van Rossum in 1991."}},        {"input": {"question": "What is RAG?"}, "output": {"answer": "RAG (Retrieval Augmented Generation) is a technique that combines document retrieval with LLM generation to ground answers in external knowledge."}},        {"input": {"question": "What is a transformer?"}, "output": {"answer": "A transformer is a neural network architecture that uses self-attention mechanisms to process sequences in parallel."}},    ]        for ex in examples:        client.create_example(inputs=ex["input"], outputs=ex["output"], dataset_id=dataset.id)        print(f"✅ Dataset '{dataset_name}' created with {len(examples)} examples")else:    print("ℹ️ Dataset creation requires LangSmith API key")    print("   Examples show:")    print("   • Creating evaluation datasets")    print("   • Adding input/output pairs")    print("   • Running evaluations against chains")

In [ ]:
# Custom evaluatorsdef correctness_evaluator(run, example):    """Check if the answer is factually correct."""    prediction = run.outputs.get("output", "")    reference = example.outputs.get("answer", "")        # Use LLM to judge correctness    judge_prompt = f"""Compare these two answers for factual correctness:    Reference: {reference}Prediction: {prediction}Are they factually consistent? Respond with just 'yes' or 'no'."""        result = llm.invoke([{"role": "user", "content": judge_prompt}])    is_correct = "yes" in result.content.lower()        return {"key": "correctness", "score": 1.0 if is_correct else 0.0}def conciseness_evaluator(run, example):    """Check if the answer is concise (under 200 chars)."""    prediction = run.outputs.get("output", "")    return {"key": "conciseness", "score": 1.0 if len(prediction) < 200 else 0.5}print("✅ Custom evaluators defined")print("   • correctness_evaluator — LLM-as-judge for factual accuracy")print("   • conciseness_evaluator — Length-based score")# Run evaluation (requires LangSmith)if client:    def predict(inputs):        result = chain.invoke({"role": "AI expert", "question": inputs["question"]})        return {"output": result}        results = evaluate(        predict,        data=dataset_name,        evaluators=[correctness_evaluator, conciseness_evaluator],        experiment_prefix="agents_course_v1",    )    print(f"\n📊 Evaluation complete — check LangSmith dashboard for results!")

---## 4. Alternatives to LangSmith| Tool | Type | Key Feature | Pricing ||------|------|-------------|---------|| **LangSmith** | SaaS + OSS | Deep LangChain integration | Free tier + paid || **Langfuse** | Open Source | Self-hosted, LangChain plugin | Free (self-hosted) || **Phoenix (Arize)** | Open Source | Great visualizations, local | Free || **OpenTelemetry** | Standard | Vendor-neutral tracing standard | Free (standard) || **Weights & Biases** | SaaS | ML experiment tracking + LLM | Free tier + paid || **Helicone** | SaaS | Proxy-based, simple setup | Free tier + paid |> **Interview Tip:** Know at least 2-3 observability tools and their trade-offs. LangSmith is the default for LangChain projects, but Langfuse and Phoenix are strong open-source alternatives.

In [ ]:
# Pattern: OpenTelemetry-style manual instrumentationimport time, functoolsdef trace_llm_call(func):    """Decorator for manual tracing without LangSmith."""    @functools.wraps(func)    def wrapper(*args, **kwargs):        start = time.time()        trace = {            "function": func.__name__,            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),            "args_preview": str(args)[:100],        }        try:            result = func(*args, **kwargs)            trace["duration_ms"] = int((time.time() - start) * 1000)            trace["status"] = "success"            trace["output_preview"] = str(result)[:100]            print(f"  📊 Trace: {func.__name__} | {trace['duration_ms']}ms | ✅")            return result        except Exception as e:            trace["duration_ms"] = int((time.time() - start) * 1000)            trace["status"] = "error"            trace["error"] = str(e)            print(f"  📊 Trace: {func.__name__} | {trace['duration_ms']}ms | ❌ {e}")            raise    return wrapper@trace_llm_calldef analyzed_call(question):    return chain.invoke({"role": "analyst", "question": question})# Testresult = analyzed_call("What is observability?")print(f"📝 {result[:150]}")

---## 📝 Interview Quiz — LangSmith & Observability### Q1: Why is observability critical for LLM applications?<details><summary>💡 Answer</summary>LLMs are **non-deterministic** — same input can produce different outputs. Without observability you can't:1. **Debug failures** — Which step in a 5-step agent failed?2. **Measure quality** — Are answers getting better or worse over time?3. **Control costs** — Which queries consume the most tokens?4. **Detect regressions** — Did a model update break anything?5. **Audit** — What did the agent decide and why? (Required for regulated industries)**Key metrics to track:** Latency (P50/P95/P99), token count, error rate, quality scores, cost per query.</details>### Q2: What's the difference between tracing and logging for LLM apps?<details><summary>💡 Answer</summary>**Logging:** Individual events (request received, error occurred)**Tracing:** Full execution path across multiple components with parent-child relationships```Trace: "user asked about RAG"     ← Trace (spans the full request)  ├── Retriever.invoke            ← Span (one step)  │     ├── Embed query           ← Sub-span  │     └── Vector search         ← Sub-span  ├── PromptTemplate.invoke       ← Span  └── ChatOpenAI.invoke           ← Span```Tracing shows **causality** — you can see that the retriever returned poor chunks, which caused the LLM to hallucinate. Logging alone can't show this.</details>### Q3: How would you set up evaluation for a RAG system?<details><summary>💡 Answer</summary>1. **Create golden dataset** — 100+ question/answer pairs with ground truth2. **Define evaluators:**   - Faithfulness: Is the answer supported by retrieved context?   - Relevance: Does the answer address the question?   - Retrieval quality: Were the right chunks retrieved?3. **Run evaluation** — Pipeline processes all examples4. **Analyze results** — Identify weak spots (retrieval vs generation)5. **CI/CD integration** — Run evals on every code change6. **A/B testing** — Compare prompt versions, models, chunking strategies</details>### Q4: LangSmith vs Langfuse — when to use each?<details><summary>💡 Answer</summary>| Aspect | LangSmith | Langfuse ||--------|-----------|----------|| Integration | Native LangChain | Via decorators/plugins || Hosting | SaaS (LangChain Inc) | Self-hosted or cloud || Privacy | Data on LangChain servers | Your infrastructure || Cost | Free tier, then paid | Free (open source) || Features | Most complete for LangChain | Good for any framework || Best for | LangChain-heavy projects | Privacy-sensitive or multi-framework |</details>

---## ✅ Notebook 21 Summary| Concept | Key Takeaway ||---------|-------------|| Tracing | Auto-records every LangChain call with inputs/outputs/latency || Datasets | Curated examples for reproducible evaluation || Evaluators | Custom functions (LLM-as-judge, heuristic) for quality scoring || Monitoring | Dashboards for latency, cost, error rate, quality trends || Alternatives | Langfuse (OSS), Phoenix (local), OpenTelemetry (standard) |### ➡️ Next: [Notebook 22 — Industry Frameworks](./22_industry_frameworks.ipynb)